# Track 3 Technical Extender

**For:** learners comfortable with Python, tests, and statistical interpretation.

Validate low-flow methods, enforce completeness, compare estimators, and design provenance. Work here or in a branch; do not overwrite the three track notebooks. **Status: method-development draft.**

## Learning objectives

By the end of the hackathon, students should be able to:

1. **Frame a relevant question** about climate, agriculture or natural-resource stewardship and explain why it interests them.
2. **Use their selected notebook pathway** to explore environmental data, documenting what they tried and any challenges encountered.
3. **Interpret and communicate evidence**, explaining the source, location, time period and meaning of any results or visualizations they present.
4. **Recognize limitations and uncertainty**, distinguishing what the data show from what would require additional evidence.
5. **Propose a future project**, identifying a next question and the data, skills or partnerships needed to pursue it.
6. **Describe potential community benefits** and explain who might find the work useful.
7. **Identify appropriate reviewers or collaborators** and explain how their perspectives could improve interpretation and guide responsible sharing.

These objectives apply across all three tracks. Students demonstrate learning through their final presentations and explanations of completed or attempted work; a finished visualization is not required.

## How we will work

Students work in **one of three tracks**, selected with mentor support: Guided Explorer, Data Investigator or Technical Extender. The tracks are parallel choices, not a sequence to complete. Begin with 60 minutes of shared instruction, followed by 15 minutes of track-group orientation. Students then start working; mentors provide brief demonstrations when a group needs them. A completed visualization is welcome but is not a condition for presenting or demonstrating learning.

Use the [seven presentation questions](../guides/final_presentation.md) to collect notes as you go.

## Agricultural application

How can regional drought and streamflow records help frame questions about livestock water availability, and what additional local evidence would we need?

Alternative questions may concern grazing, gardens, plant resources or watershed stewardship. State what additional evidence would be needed; regional indicators alone do not establish a management recommendation.

## 1. Test the low-flow definition

A rolling seven-day minimum of daily values and a rolling seven-day mean answer different questions. Test the distinction before applying either method.

## Sovereignty activity 1: who shapes the question? (3–5 minutes)

Data sovereignty concerns Indigenous Peoples' authority over data relationships and uses. Data governance is how decisions about collection, interpretation, access and reuse are put into practice. For this exercise, the question itself is a decision: who chose it, whose priorities does it reflect, and who could change it?

With your group, identify a possible benefit, a role or body whose direction would be needed for a real project, and one kind of information you will **not** collect here. Do not speak on behalf of a Nation or invent approval. You can discuss a hypothetical situation without sharing personal, cultural or protected knowledge.

Start the decision notes below. Use role descriptions, not private contact details. `None` means unresolved, not permission. Keep sensitive answers outside this notebook in the appropriate setting. See [sovereignty practice](../guides/sovereignty_practice.md) for optional framework references.


In [ ]:
# CUSTOMIZE: brief, non-sensitive discussion notes. No approval is inferred.
governance_notes = {
    "question": None,
    "potential_benefit": None,
    "authority_to_consult": None,
    "community_input_needed": None,
    "representation_limits": None,
    "reviewer_roles": None,
    "intended_audience": "agreed classroom audience; no public release assumed",
    "storage_and_access": None,
    "reuse_limits": "classroom exercise; revisit purpose and permissions before reuse",
    "correction_withdrawal": None,
    "decision": None,
    "decision_reason": None,
}
print("Start with question, benefit and authority to consult. Leave unknowns as None.")

In [ ]:
import json
from pathlib import Path
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
start = Path.cwd().resolve()
REPO = next((p for p in [start, *start.parents] if (p / 'data').is_dir()), None)
DATA = REPO/'data'/'sample_or_fallback' if REPO else None
print(f'Repository: {REPO}')

In [ ]:
def rolling_daily_minimum(values, window=7):
    return pd.Series(values, dtype=float).rolling(window, min_periods=window).min()

def rolling_average(values, window=7):
    return pd.Series(values, dtype=float).rolling(window, min_periods=window).mean()

hand_series = [10, 10, 10, 1, 10, 10, 10]
assert rolling_daily_minimum(hand_series).iloc[-1] == 1
assert np.isclose(rolling_average(hand_series).iloc[-1], 61 / 7)
print('PASS: the two methods differ on the hand-calculated series.')

## 2. Load a gauge and enforce completeness

The ≥330 observed-days rule below is an explicit teaching choice to evaluate.

In [ ]:
SITE_ID = '06446000'
matches = sorted(DATA.glob(f'usgs_nwis_{SITE_ID}_*.rdb')) if DATA else []
if not matches:
    raise FileNotFoundError('Prepared NWIS snapshot missing.')
lines = [line for line in matches[-1].read_text(encoding='utf-8').splitlines() if not line.startswith('#')]
header = lines[0].split('\t')
rows = [line for line in lines[2:] if line.startswith('USGS')]
raw = pd.read_csv(StringIO('\n'.join(rows)), sep='\t', header=None, names=header, dtype=str)
value_column = next(c for c in raw.columns if '00060' in c and not c.endswith('_cd'))
flow = pd.DataFrame({'date': pd.to_datetime(raw['datetime'], errors='coerce'), 'flow_cfs': pd.to_numeric(raw[value_column], errors='coerce')}).dropna().sort_values('date')
flow['year'] = flow['date'].dt.year
flow['flow_7day_mean'] = flow.set_index('date')['flow_cfs'].rolling('7D', min_periods=7).mean().to_numpy()
annual = flow.groupby('year').agg(observed_days=('date','nunique'), annual_7day_low=('flow_7day_mean','min')).reset_index()
annual['usable'] = annual['observed_days'] >= 330
fitted = annual[annual['usable']].dropna(subset=['annual_7day_low'])
assert (fitted['observed_days'] >= 330).all()
print(f'PASS {len(fitted)} usable years; {len(annual)-len(fitted)} excluded years.')
annual.tail()

## Sovereignty activity 2: what does this data represent? (3–5 minutes)

A correct calculation can still answer the wrong question or be used without appropriate direction. Who should decide whether a low-flow measure is relevant to a local use? What is lost when qualifier codes or missing periods are ignored? Explain how code tests, hydrologic review and community direction each contribute different evidence. Propose a design change that keeps unresolved governance decisions visible when results are exported.

Update `representation_limits` and `community_input_needed` below. A data gap is an unanswered question; it does not establish lack of community knowledge.

In [ ]:
# CUSTOMIZE after discussing representation. Use non-sensitive notes only.
governance_notes["representation_limits"] = None
governance_notes["community_input_needed"] = None

## 3. Compare estimators

The implementation and narrative must name the same method. Neither fitted line establishes causality.

In [ ]:
x = fitted['year'].to_numpy(dtype=float)
y = fitted['annual_7day_low'].to_numpy(dtype=float)
ols = stats.linregress(x, y)
theil = stats.theilslopes(y, x, alpha=0.95)
comparison = pd.DataFrame([
 {'method':'Ordinary least squares','slope_cfs_decade':ols.slope*10,'lower_95':np.nan,'upper_95':np.nan,'p_value':ols.pvalue},
 {'method':'Theil–Sen','slope_cfs_decade':theil.slope*10,'lower_95':theil.low_slope*10,'upper_95':theil.high_slope*10,'p_value':np.nan}
])
display(comparison)
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(x,y,label='Annual 7-day low')
ax.plot(x,ols.intercept+ols.slope*x,label='OLS')
ax.plot(x,theil.intercept+theil.slope*x,'--',label='Theil–Sen')
ax.set(xlabel='Year',ylabel='Annual minimum 7-day mean flow (cfs)',title=f'Method comparison USGS {SITE_ID}')
ax.legend(); plt.show()

## Sovereignty activity 3: choose the next use (5 minutes)

Imagine a request to post your chart online or reuse it in a project about a different community. What changes in purpose, audience or representation would need review? Discuss an option to revise, limit or decline the proposed use. Who can ask for a correction or withdrawal, and who would act on it?

Record your decision and reason, potential benefit, reviewer roles, storage/access and correction process. You may leave unresolved items as `None` and explain what needs to happen next. No student is expected to provide consent on behalf of a Nation. Keep real sensitive information out of this exercise.


In [ ]:
# CUSTOMIZE: a proposed classroom decision, not an authorization.
governance_notes.update({
    "potential_benefit": None,
    "authority_to_consult": None,
    "reviewer_roles": None,
    "storage_and_access": None,
    "correction_withdrawal": None,
    "decision": None,  # e.g., revise the claim; keep within class; seek review before reuse
    "decision_reason": None,
})

## Provenance: technical lineage and governance decisions

This extends the Technical Extender provenance exercise to every track. The cell records the actual snapshot URLs/dates and verified checksums, notebook identity, analytical settings, limitations and your discussion notes. Review these together: technical correctness alone does not settle authority or appropriate use.

The [IEEE 2890-2025 recommended practice](https://standards.ieee.org/ieee/2890/10318/) addresses provenance relevant to Indigenous Peoples' data relationships and governance. This classroom record explores those ideas; it is not a standards-conformity assessment or evidence of Tribal endorsement. The [toolkit and framework references](../guides/sovereignty_practice.md) provide further reading.


In [ ]:
import sys, json
sys.path.insert(0, str(REPO / "src"))
from provenance import build_provenance, save_draft
source_files = [matches[-1].name]
analysis = {
    "source_steward": "USGS", "site_id": SITE_ID,
    "measure": "annual minimum of rolling seven-day mean daily discharge",
    "units": "cubic feet per second; trend slopes in cfs per decade",
    "returned_period": [str(flow["date"].min().date()), str(flow["date"].max().date())],
    "fitted_years": [int(y) for y in fitted["year"]],
    "completeness_rule": "seven observations in a seven-day window; at least 330 observed days/calendar year",
    "estimators": ["ordinary least squares", "Theil–Sen"],
    "processing": ["parse RDB and remove missing date/value rows", "sort dates", "calculate seven-day means", "annual minima", "filter completeness", "compare estimators"],
    "limitations": ["qualifier-code review remains needed", "local hydrologic interpretation required", "annual minima may use windows crossing year boundaries", "not a 7Q10 recurrence statistic", "regression assumptions and governance remain subject to review"],
    "artifact": "in-notebook low-flow estimator comparison",
}
provenance = build_provenance(REPO, "tracks/03_technical_extender.ipynb", source_files, analysis, governance_notes)
print(json.dumps(provenance, indent=2))
print("Unresolved discussion fields:", provenance["unresolved_fields"])
# Completing fields does not change the classroom-draft status.


### Optional: save a local draft record

If your notes contain only appropriate classroom information, set `SAVE_DRAFT=True` to save a JSON sidecar under ignored `outputs/governance/`. It describes the in-notebook artifacts; it does not export a figure or publish anything. Use the agreed class storage for completed work. The notebook itself can also retain edited notes—clear private information before sharing the notebook. Re-run the provenance cell after changing analysis settings or notes.


In [ ]:
SAVE_DRAFT = False
if SAVE_DRAFT:
    draft_path = save_draft(provenance, REPO / "outputs" / "governance")
    print("Saved local classroom draft:", draft_path)
else:
    print("Draft displayed above; no sidecar saved.")

## What to bring to the final presentation

Explain what you tried and learned; show any visualizations and describe their source and meaning. Record uncertainty and future project ideas. A completed figure is not required. Use the seven questions below to prepare.

## Final presentation and stewardship

1. **What question did you explore, and why did it interest you?**
2. **What did you learn?** Describe something about the topic, data or method.
3. **What did you create?** Show any visualizations and explain their source, place, period and meaning. If you did not finish a visualization, explain what you tried and what happened.
4. **What remains uncertain?** Identify a limitation, challenge or unanswered question.
5. **What would you investigate next?** Suggest a future project and the data, skills or partnerships it would need.
6. **Who might benefit from this work?** Explain the potential benefit without claiming an outcome the project has not demonstrated.
7. **Who should review or help interpret this work?** Identify relevant people or roles and why their perspective matters; naming a reviewer does not imply approval.

See the [presentation guidance](../guides/final_presentation.md). Save full stewardship details in the agreed class location. The final hour, September 16 10:45–11:45, is for student sharing and questions.

### Bring your decisions into the seven-question presentation

Use the provenance record to explain sources and limitations (questions 3–4). Explain a future-use decision (question 5), a possible benefit (question 6) and the reviewer roles/perspectives needed (question 7). You can state that a decision remains unresolved. Do not show private details or claim approval that has not occurred.
